# Setup and Dependencies


In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install timm efficientnet-pytorch
!pip install kaggle pandas scikit-learn opencv-python tqdm seaborn
!pip install torchmetrics
!pip install pydicom
!pip install --upgrade kaggle

import sys
print("✓ Dependencies installed!")
print(f"Python version: {sys.version}")

import torch
print(f"PyTorch version: {torch.__version__}")
# Use torch.version.cuda to get the CUDA toolkit version
print(f"CUDA toolkit version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Looking in indexes: https://download.pytorch.org/whl/cu118
✓ Dependencies installed!
Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch version: 2.10.0+cu128
CUDA toolkit version: 12.8
GPU: NVIDIA L4


# Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create directory for saving models
import os
os.makedirs('/content/drive/MyDrive/EyeShield/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/EyeShield/logs', exist_ok=True)

print("✓ Google Drive mounted!")
print("Save location: /content/drive/MyDrive/EyeShield/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted!
Save location: /content/drive/MyDrive/EyeShield/


# Setup Kaggle API

In [ ]:
from google.colab import files
import json
from pathlib import Path

print("Upload your kaggle.json file...")
print("Instructions:")
print("1. Go to: https://www.kaggle.com/settings/account")
print("2. Click 'Create New API Token'")
print("3. Upload the downloaded kaggle.json file")

uploaded = files.upload()

if 'kaggle.json' in uploaded:
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(exist_ok=True)

    with open(kaggle_dir / 'kaggle.json', 'w') as f:
        f.write(uploaded['kaggle.json'].decode())

    os.chmod(kaggle_dir / 'kaggle.json', 0o600)
    print("✓ Kaggle API configured!")
else:
    print("⚠ kaggle.json not found. You can continue with sample data.")

Upload your kaggle.json file...
Instructions:
1. Go to: https://www.kaggle.com/settings/account
2. Click 'Create New API Token'
3. Upload the downloaded kaggle.json file


Saving kaggle (1).json to kaggle (1) (1).json
⚠ kaggle.json not found. You can continue with sample data.


# Download Dataset from Kaggle

In [ ]:
import kagglehub

# Unzip if needed
import zipfile
dataset_path = kagglehub.dataset_download("ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy")

for file in os.listdir(dataset_path):
    if file.endswith('.zip'):
        with zipfile.ZipFile(os.path.join(dataset_path, file), 'r') as zip_ref:
            zip_ref.extractall(dataset_path)

print(f"✓ Dataset downloaded to: {dataset_path}")
print(f"Files: {os.listdir(dataset_path)[:10]}")


Using Colab cache for faster access to the 'eyepacs-aptos-messidor-diabetic-retinopathy' dataset.
✓ Dataset downloaded to: /kaggle/input/eyepacs-aptos-messidor-diabetic-retinopathy
Files: ['dr_unified_v2', 'augmented_resized_V2']


# Copy Training Script

In [ ]:
!wget -O /content/eyeshield_training_preprocessor.py \
  "https://raw.githubusercontent.com/dondondon22/EyeShield/refs/heads/main/eyeshield_training_preprocessor.py"

# OR if using GitHub Gist:
# !wget -O /content/eyeshield_sprint3_training.py "https://gist.githubusercontent.com/..."

print("✓ Training script downloaded!")

--2026-03-06 10:29:44--  https://raw.githubusercontent.com/dondondon22/EyeShield/refs/heads/main/eyeshield_training_preprocessor.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 31320 (31K) [text/plain]
Saving to: ‘/content/eyeshield_training_preprocessor.py’

/content/eyeshield_ 100%[===================>]  30.59K  --.-KB/s    in 0.002s  

2026-03-06 10:29:45 (13.6 MB/s) - ‘/content/eyeshield_training_preprocessor.py’ saved [31320/31320]

✓ Training script downloaded!


# Prepare Dataset CSV

In [ ]:
import pandas as pd
import os
from pathlib import Path

# The dataset was downloaded to `dataset_path` variable in the previous step.
# Use this variable for the root directory of the dataset.
dataset_root = kagglehub.dataset_download("ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy") # Re-define for isolated execution

# Create directory for the CSV file if it doesn't exist
os.makedirs('/content/dataset', exist_ok=True)

images = []
labels = []

print(f"DEBUG: Dataset root: {dataset_root}")
print(f"DEBUG: Contents of dataset root: {os.listdir(dataset_root)}")

# The dataset contains subdirectories like 'dr_unified_v2' and 'augmented_resized_V2'
# We need to iterate through these to find the actual image folders (0, 1, 2, 3, 4)
for sub_dir in os.listdir(dataset_root):

    current_dataset_path = os.path.join(dataset_root, sub_dir)
    if os.path.isdir(current_dataset_path):
        print(f"DEBUG:   Processing subdirectory: {current_dataset_path}")
        # Look for 'train', 'test', 'validation' directories
        for data_split in os.listdir(current_dataset_path):
            data_split_path = os.path.join(current_dataset_path, data_split)
            if os.path.isdir(data_split_path):
                print(f"DEBUG:     Processing data split: {data_split_path}")
                # Now look for the actual DR level directories (0, 1, 2, 3, 4)
                for dr_level in os.listdir(data_split_path):
                    level_dir = os.path.join(data_split_path, dr_level)
                    if os.path.isdir(level_dir):
                        try:
                            level_int = int(dr_level)
                            # print(f"DEBUG:       Processing DR level directory: {level_dir}")
                            found_images_in_level = False
                            for img_file in os.listdir(level_dir):
                                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                                    full_img_path = os.path.join(level_dir, img_file)
                                    images.append(os.path.relpath(full_img_path, dataset_root))
                                    labels.append(level_int)
                                    found_images_in_level = True
                            if not found_images_in_level:
                                print(f"DEBUG:         No images found in {level_dir} or they don't match extensions.")
                        except ValueError:
                            print(f"DEBUG:       Skipping non-integer directory: {level_dir}")
                    else:
                        print(f"DEBUG:       {level_dir} is not a directory. Skipping.")
            else:
                print(f"DEBUG:     {data_split_path} is not a directory. Skipping.")
    else:
        print(f"DEBUG:   dr_unified_v2 directory not found at {current_dataset_path}")

df = pd.DataFrame({
    'image_path': images,
    'diagnosis': labels
})

df.to_csv('/content/dataset/labels.csv', index=False)

print(f"✓ Dataset CSV created!")
print(f"Total images: {len(df)}")
print(f"Class distribution:\n{df['diagnosis'].value_counts().sort_index()}")
print(f"\nSample:\n{df.head()}")

IndentationError: expected an indented block after 'for' statement on line 20 (2462179732.py, line 22)

# Modify Config and Run Training

In [ ]:
# Read the training script
with open('/content/eyeshield_training_preprocessor.py', 'r') as f:
    training_code = f.read()

# Modify paths in the Config class (if needed)
modified_code = training_code.replace(
    "CHECKPOINT_DIR = './checkpoints'",
    "CHECKPOINT_DIR = '/content/drive/MyDrive/EyeShield/checkpoints'"
)

modified_code = modified_code.replace(
    "LOG_DIR = './logs'",
    "LOG_DIR = '/content/drive/MyDrive/EyeShield/logs'"
)

# Save modified script
with open('/content/eyeshield_training_preprocessor_modified.py', 'w') as f:
    f.write(modified_code)

print("✓ Configuration updated!")
print("Ready to start training...")

✓ Configuration updated!
Ready to start training...


# Backup

In [ ]:

# Setup Auto-Backup on Colab Interruption
import shutil
import signal
import atexit
from datetime import datetime

backup_executed = False

def backup_all_files():
    """Backup CSV and training artifacts to Google Drive"""
    global backup_executed
    if backup_executed:
        return

    backup_executed = True
    print("\n" + "="*80)
    print("EXECUTING BACKUP TO GOOGLE DRIVE")
    print("="*80)

    try:
        # Backup CSV to Drive
        csv_source = '/content/dataset/labels.csv'
        csv_dest = '/content/drive/MyDrive/EyeShield/labels_backup.csv'

        if os.path.exists(csv_source):
            shutil.copy2(csv_source, csv_dest)
            print(f"✓ Backed up CSV: {csv_dest}")
        else:
            print(f"⚠ CSV not found at {csv_source}")

        # Backup logs directory
        logs_source = '/content/logs'
        logs_dest = '/content/drive/MyDrive/EyeShield/logs_backup'

        if os.path.exists(logs_source):
            if os.path.exists(logs_dest):
                shutil.rmtree(logs_dest)
            shutil.copytree(logs_source, logs_dest)
            print(f"✓ Backed up logs: {logs_dest}")

        # Backup modified training script
        script_source = '/content/eyeshield_training_preprocessor_modified.py'
        script_dest = '/content/drive/MyDrive/EyeShield/training_script_backup.py'

        if os.path.exists(script_source):
            shutil.copy2(script_source, script_dest)
            print(f"✓ Backed up training script: {script_dest}")

        print(f"Backup timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("="*80 + "\n")

    except Exception as e:
        print(f"❌ Backup error: {e}")
        import traceback
        traceback.print_exc()

def signal_handler(sig, frame):
    """Handle interruption signal"""
    print("\n⚠ Colab interruption detected! Running backup...")
    backup_all_files()
    raise KeyboardInterrupt

# Register backup to run on exit
atexit.register(backup_all_files)

# Register signal handlers for interruption
signal.signal(signal.SIGTERM, signal_handler)
signal.signal(signal.SIGINT, signal_handler)

print("✓ Auto-backup enabled on Colab stop/interruption")


#  DEbug Training

In [ ]:
# Debug: Verify all dependencies and paths exist

import os
import sys

print("=" * 80)
print("PRE-TRAINING CHECKS")
print("=" * 80)

# Check Python version
print(f"\n1. Python Version: {sys.version}")

# Check required packages
required_packages = ['torch', 'pandas', 'cv2', 'PIL', 'pydicom', 'sklearn']
print(f"\n2. Checking packages:")
for pkg in required_packages:
    try:
        __import__(pkg)
        print(f"   ✓ {pkg} available")
    except ImportError:
        print(f"   ✗ {pkg} NOT AVAILABLE")

# Check file paths
print(f"\n3. Checking file paths:")
files_to_check = [
    '/content/eyeshield_training_preprocessor_modified.py',
    '/content/dataset/labels.csv',
    '/content/drive/MyDrive/EyeShield/checkpoints',
    '/content/drive/MyDrive/EyeShield/logs'
]

for path in files_to_check:
    exists = os.path.exists(path)
    status = "✓" if exists else "✗"
    print(f"   {status} {path}")

# Check dataset CSV
try:
    import pandas as pd
    df = pd.read_csv('/content/dataset/labels.csv')
    print(f"\n4. Dataset CSV:")
    print(f"   ✓ Loaded {len(df)} records")
    print(f"   Columns: {list(df.columns)}")
    print(f"   Class distribution:\n{df['diagnosis'].value_counts().sort_index()}")
except Exception as e:
    print(f"\n4. Dataset CSV: ✗ Error - {e}")

print("\n" + "=" * 80)


PRE-TRAINING CHECKS

1. Python Version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]

2. Checking packages:
   ✓ torch available
   ✓ pandas available
   ✓ cv2 available
   ✓ PIL available
   ✓ pydicom available
   ✓ sklearn available

3. Checking file paths:
   ✓ /content/eyeshield_training_preprocessor_modified.py
   ✓ /content/dataset/labels.csv
   ✓ /content/drive/MyDrive/EyeShield/checkpoints
   ✓ /content/drive/MyDrive/EyeShield/logs

4. Dataset CSV:
   ✓ Loaded 0 records
   Columns: ['image_path', 'diagnosis']
   Class distribution:
Series([], Name: count, dtype: int64)



In [ ]:

# Manual Backup (Run Anytime)
backup_all_files()
print("Manual backup completed. You can run this cell anytime to backup current progress.")


# TRAIN

In [ ]:
# Execute the full training pipeline

%cd /content

import sys
sys.path.insert(0, '/content')

# Import all training components
print("Loading training script...")
try:
    # Read and execute the modified training script
    with open('/content/eyeshield_training_preprocessor_modified.py', 'r') as f:
        training_script = f.read()

    # Create a clean namespace for execution
    exec_namespace = {'__name__': '__main__', '__file__': '/content/eyeshield_training_preprocessor_modified.py'}

    # Execute the script
    exec(training_script, exec_namespace)

    print("\n✓ Training completed!")

except FileNotFoundError as e:
    print(f"❌ File not found: {e}")
    print("   Make sure the 'Modify Config and Run Training' cell was executed first")
except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {str(e)}")
    import traceback
    print("\nFull traceback:")
    traceback.print_exc()

finally:
    # Always backup on training completion or error
    print("\n⏳ Running final backup...")
    backup_all_files()
    print("✓ All files backed up to Google Drive")

/content
Loading training script...
Device: cuda
CUDA Available: True
GPU: NVIDIA A100-SXM4-40GB

EyeShield: DR Classification Model Training (Sprint 3)
Using Your Image Preprocessor (No CLAHE)
Configuration:
  - Num Classes: 5
  - Target Preprocessing Size: (640, 640)
  - Input Size: (640, 640)
  - Batch Size: 64
  - Num Epochs: 150
  - Learning Rate: 0.0001
  - EDL KL Weight: 0.1
  - Quality Check: True

Initializing image preprocessor...
✓ Image preprocessor initialized
  - Target size: (640, 640)
  - Quality assessment: Enabled

Loading dataset from CSV...
✓ Loaded 0 images from dataset
  - Class distribution:
Series([], Name: count, dtype: int64)

Data split:
  - Train: 0 images
  - Val: 0 images
  - Test: 0 images

❌ Error: ValueError: num_samples should be a positive integer value, but got num_samples=0

Full traceback:


Traceback (most recent call last):
  File "/tmp/ipykernel_1600/144160113.py", line 19, in <cell line: 0>
    exec(training_script, exec_namespace)
  File "<string>", line 862, in <module>
  File "<string>", line 822, in main
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 394, in __init__
    sampler = RandomSampler(dataset, generator=generator)  # type: ignore[arg-type]
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/sampler.py", line 149, in __init__
    raise ValueError(
ValueError: num_samples should be a positive integer value, but got num_samples=0


# RESUME TRAINING

In [ ]:
# Resume Training from Best Checkpoint
import os
import glob
import torch

checkpoint_dir = '/content/drive/MyDrive/EyeShield/checkpoints'

print("="*80)
print("RESUME TRAINING FROM CHECKPOINT")
print("="*80)

# List available checkpoints
checkpoint_files = glob.glob(os.path.join(checkpoint_dir, '*.pt'))
checkpoint_files.sort(key=os.path.getmtime)

print(f"\n📁 Checkpoint directory: {checkpoint_dir}")
print(f"Found {len(checkpoint_files)} checkpoint(s):\n")

best_model_path = os.path.join(checkpoint_dir, 'best_model.pt')
if os.path.exists(best_model_path):
    # Get modification time
    import datetime
    mtime = os.path.getmtime(best_model_path)
    mtime_str = datetime.datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M:%S')
    size_mb = os.path.getsize(best_model_path) / (1024**2)
    print(f"✓ best_model.pt (BEST) - Modified: {mtime_str}, Size: {size_mb:.2f}MB")

for cp in checkpoint_files:
    if 'best_model' not in cp:
        mtime = os.path.getmtime(cp)
        mtime_str = datetime.datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M:%S')
        size_mb = os.path.getsize(cp) / (1024**2)
        print(f"  • {os.path.basename(cp)} - Modified: {mtime_str}, Size: {size_mb:.2f}MB")

# Load best model checkpoint
print(f"\n⏳ Loading best_model.pt...")
try:
    checkpoint = torch.load(best_model_path, map_location='cpu')
    
    print(f"✓ Checkpoint loaded successfully!")
    print(f"  - Trained for {checkpoint.get('epoch', 'unknown')} epochs")
    print(f"  - Validation metrics: {checkpoint.get('val_metrics', {})}")
    
    # Store checkpoint for use in training cell
    RESUME_CHECKPOINT = checkpoint
    RESUME_CHECKPOINT_PATH = best_model_path
    RESUME_FROM_EPOCH = checkpoint.get('epoch', 0) + 1
    
    print(f"\n✓ Ready to resume from epoch {RESUME_FROM_EPOCH}")
    print("  Run the training cell below to continue from this checkpoint")
    
except Exception as e:
    print(f"❌ Error loading checkpoint: {e}")
    RESUME_CHECKPOINT = None


# Resume TRAINING

In [ ]:
# RESUME TRAINING from Checkpoint
# Make sure to run the previous cell first to load the checkpoint

%cd /content

if 'RESUME_CHECKPOINT' in locals() and RESUME_CHECKPOINT is not None:
    print("="*80)
    print("RESUMING TRAINING FROM CHECKPOINT")
    print("="*80)
    
    import sys
    sys.path.insert(0, '/content')
    
    try:
        # Read and execute the modified training script
        with open('/content/eyeshield_training_preprocessor_modified.py', 'r') as f:
            training_script = f.read()

        # Create a clean namespace for execution
        exec_namespace = {
            '__name__': '__main__',
            '__file__': '/content/eyeshield_training_preprocessor_modified.py',
            'RESUME_CHECKPOINT': RESUME_CHECKPOINT,
            'RESUME_FROM_EPOCH': RESUME_FROM_EPOCH,
            'RESUME_CHECKPOINT_PATH': RESUME_CHECKPOINT_PATH
        }

        # Inject resume logic into the training script
        resume_injection = """
# ============ CHECKPOINT RESUME INJECTION ============
if 'RESUME_CHECKPOINT' in locals() and RESUME_CHECKPOINT is not None:
    print("\\n" + "="*80)
    print("LOADING BEST CHECKPOINT FOR RESUME")
    print("="*80)
    
    # Load checkpoint
    resume_ckpt = RESUME_CHECKPOINT
    trainer.model.load_state_dict(resume_ckpt['model_state'])
    trainer.optimizer.load_state_dict(resume_ckpt['optimizer_state'])
    resume_epoch = resume_ckpt.get('epoch', 0)
    
    print(f"✓ Loaded checkpoint from epoch {resume_epoch}")
    print(f"✓ Model weights restored")
    print(f"✓ Optimizer state restored")
    print(f"\\nResuming training from epoch {resume_epoch + 1}...")
    print("="*80 + "\\n")
    
    # Override starting epoch
    start_epoch = resume_epoch + 1
else:
    start_epoch = 0
# ====================================================
"""
        
        # Find where to inject (after trainer initialization)
        if 'trainer = Trainer' in training_script:
            inject_pos = training_script.find('trainer.train()')
            if inject_pos > 0:
                training_script = training_script[:inject_pos] + resume_injection + training_script[inject_pos:]

        # Execute the script with resume capability
        exec(training_script, exec_namespace)

        print("\n✓ Training resumed and completed!")

    except FileNotFoundError as e:
        print(f"❌ File not found: {e}")
        print("   Make sure the 'Modify Config and Run Training' cell was executed first")
    except Exception as e:
        print(f"❌ Error: {type(e).__name__}: {str(e)}")
        import traceback
        print("\nFull traceback:")
        traceback.print_exc()

    finally:
        # Always backup on completion
        print("\n⏳ Running final backup...")
        backup_all_files()
        print("✓ All files backed up to Google Drive")

else:
    print("❌ No checkpoint loaded!")
    print("   Please run the 'Resume Training from Best Checkpoint' cell first to load a checkpoint")


In [ ]:
# Checkpoint Inspector (Optional - View Details Before Resuming)
import torch
import os
import json

checkpoint_path = '/content/drive/MyDrive/EyeShield/checkpoints/best_model.pt'

if os.path.exists(checkpoint_path):
    print("="*80)
    print("CHECKPOINT DETAILS")
    print("="*80)
    
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    
    print(f"\n📋 Checkpoint Information:")
    print(f"   Path: {checkpoint_path}")
    print(f"   File size: {os.path.getsize(checkpoint_path) / (1024**2):.2f} MB")
    
    if 'epoch' in checkpoint:
        print(f"   Last trained epoch: {checkpoint['epoch']}")
    
    if 'val_metrics' in checkpoint:
        metrics = checkpoint['val_metrics']
        print(f"\n📊 Validation Metrics at last save:")
        for key, value in metrics.items():
            if isinstance(value, (int, float)):
                print(f"      {key}: {value:.4f}")
    
    if 'model_state' in checkpoint:
        model_size = sum(p.numel() for p in checkpoint['model_state'].values())
        print(f"\n🧠 Model:")
        print(f"      Total parameters: {model_size:,}")
    
    print(f"\n✓ This checkpoint is ready to resume from!")
    print(f"  Next epoch will be: {checkpoint.get('epoch', 0) + 1}")
    
else:
    print(f"❌ Checkpoint not found at: {checkpoint_path}")
    print("   Train a model first or check the path")


# Monitor Training (Optional)

In [ ]:
# Use tensorboard to monitor training in real-time

# %load_ext tensorboard
# %tensorboard --logdir /content/logs

# Download Results

In [ ]:
from google.colab import files

# Download best model
files.download('/content/drive/MyDrive/EyeShield/checkpoints/best_model.pt')

# Download training history plot
files.download('/content/drive/MyDrive/EyeShield/logs/training_history.png')

# Download all checkpoints as zip
!cd /content/drive/MyDrive/EyeShield && zip -r checkpoints.zip checkpoints/
files.download('/content/drive/MyDrive/EyeShield/checkpoints.zip')

print("✓ Results downloaded!")

# Evaluation and Testing

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# Load best model
checkpoint = torch.load('/content/drive/MyDrive/EyeShield/checkpoints/best_model.pt')
model = EfficientNetB3EDL(num_classes=5, pretrained=False)
model.load_state_dict(checkpoint['model_state'])
model.eval()

# Create test loader (using validation set as proxy)
# ... your test loading code here ...

# Evaluate
all_preds = []
all_targets = []
all_uncertainties = []

with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device)
        targets = targets.to(device)

        evidence = model(images)
        output = model.predict(evidence)

        all_preds.extend(output['pred'].cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
        all_uncertainties.extend(output['uncertainty'].cpu().numpy())

# Confusion Matrix
cm = confusion_matrix(all_targets, all_preds)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative'],
            yticklabels=['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative'])
ax.set_title('Confusion Matrix - DR Classification')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/EyeShield/logs/confusion_matrix.png', dpi=300)
plt.show()

# Classification Report
print("Classification Report:")
print(classification_report(all_targets, all_preds,
      target_names=['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']))

# Uncertainty Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(all_uncertainties, bins=50, alpha=0.7)
axes[0].set_title('Uncertainty Distribution')
axes[0].set_xlabel('Uncertainty')
axes[0].set_ylabel('Frequency')

# Correct vs Incorrect predictions
correct_unc = [u for u, p, t in zip(all_uncertainties, all_preds, all_targets) if p == t]
incorrect_unc = [u for u, p, t in zip(all_uncertainties, all_preds, all_targets) if p != t]

axes[1].hist(correct_unc, bins=30, alpha=0.7, label='Correct')
axes[1].hist(incorrect_unc, bins=30, alpha=0.7, label='Incorrect')
axes[1].set_title('Uncertainty: Correct vs Incorrect')
axes[1].set_xlabel('Uncertainty')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/EyeShield/logs/uncertainty_analysis.png', dpi=300)
plt.show()

print("✓ Evaluation complete!")